# Applied AI Midterm SRGAN — Complete Colab Pipeline
Run this notebook from top to bottom in one GPU-backed Colab session. All long-running outputs and checkpoints are linked to Google Drive, so the SRGAN stage can resume after a runtime disconnect. Do not skip phase cells or use the reserved test split for training.

---

# Phase 0 — Colab and persistent Drive setup


# Google Colab setup
Before running this notebook, choose **Runtime → Change runtime type → T4 GPU**. This notebook mounts Drive, clones or updates the repository, installs the package without replacing Colab's GPU-enabled PyTorch, links persistent experiment storage, verifies the dataset, and creates/reuses the seed-42 split.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = "https://github.com/jtackerman1/Applied_AI_Midterm_SRGAN.git"
REPOSITORY_DIR = Path("/content/Applied_AI_Midterm_SRGAN")
if (REPOSITORY_DIR / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "PyYAML>=6.0.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPOSITORY_DIR), "--no-deps"], check=True)
print(f"Repository ready at {REPOSITORY_DIR}")

In [ ]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled. Select Runtime → Change runtime type → T4 GPU, then rerun.")
print(f"GPU: {torch.cuda.get_device_name(0)}")

## Persistent Drive layout
Upload the local `data/raw` directory to `MyDrive/Applied_AI_Midterm_SRGAN_runtime/data/raw`. At minimum, Colab must see `data/raw/train/cats` and `data/raw/train/dogs`. Vendor `validation` and `test` folders may also be present, but the project intentionally ignores them and creates its own 70/30 split.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/Applied_AI_Midterm_SRGAN_runtime")
persistent_links = {
    REPOSITORY_DIR / "data/raw": DRIVE_ROOT / "data/raw",
    REPOSITORY_DIR / "data/splits": DRIVE_ROOT / "data/splits",
    REPOSITORY_DIR / "data/generated": DRIVE_ROOT / "data/generated",
    REPOSITORY_DIR / "checkpoints": DRIVE_ROOT / "checkpoints",
    REPOSITORY_DIR / "artifacts": DRIVE_ROOT / "artifacts",
}
for link_path, drive_path in persistent_links.items():
    drive_path.mkdir(parents=True, exist_ok=True)
    if link_path.is_symlink():
        link_path.unlink()
    elif link_path.exists():
        shutil.rmtree(link_path)
    link_path.parent.mkdir(parents=True, exist_ok=True)
    link_path.symlink_to(drive_path, target_is_directory=True)
    print(f"{link_path.relative_to(REPOSITORY_DIR)} -> {drive_path}")

In [ ]:
from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import discover_images, prepare_splits

config = load_config(REPOSITORY_DIR / "configs/config.yaml")
try:
    discovered = discover_images(REPOSITORY_DIR / "data/raw")
except (FileNotFoundError, ValueError) as error:
    raise RuntimeError(
        "Dataset is not ready in Drive. Upload it to "
        f"{DRIVE_ROOT / 'data/raw'} and rerun this cell. Original error: {error}"
    ) from error
display(discovered.groupby(["class_name", "label"]).size().rename("source_images").to_frame())
splits = prepare_splits(
    REPOSITORY_DIR / "data/raw",
    REPOSITORY_DIR / "data/splits",
    train_ratio=config.train_ratio,
    random_seed=config.random_seed,
)
display({"training_examples": len(splits.train), "reserved_test_examples": len(splits.test)})

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPOSITORY_DIR, check=True)
print("Setup complete. Continue with the phase cells below in this notebook.")

## Combined execution and resume order
1. `01_data_preparation.ipynb` — inspect the already persisted split.
2. `02_model_a_transfer_learning.ipynb` — train Model A.
3. `03_srgan_training.ipynb` — pretrain and adversarially train SRGAN. This is the long stage; rerun setup and notebook 03 after a disconnect to resume from Drive checkpoints.
4. `04_generate_srgan_training_images.ipynb` — generate training images and manifest.
5. `05_model_b_transfer_learning.ipynb` — train Model B.
6. `06_final_comparison.ipynb` — produce measured results.

Do not delete `MyDrive/Applied_AI_Midterm_SRGAN_runtime`; it contains all checkpoints and outputs.

---

# Phase 1 — Dataset preparation


# Dataset preparation
Discover the binary dataset, create or reuse the persistent 70/30 stratified split, and inspect the preprocessing pipelines. Re-running this notebook reuses existing CSV files rather than reshuffling examples.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import discover_images, prepare_splits
from applied_ai_midterm.transforms import (
    SRGANPairTransform,
    classifier_transform,
    denormalize_imagenet,
    denormalize_srgan,
)

config = load_config(Path("configs/config.yaml"))
raw_data_dir = Path("data/raw")
splits_dir = Path("data/splits")

In [ ]:
discovered = discover_images(raw_data_dir)
display(discovered.groupby(["class_name", "label"]).size().rename("image_count").to_frame())

In [ ]:
splits = prepare_splits(
    raw_data_dir,
    splits_dir,
    train_ratio=config.train_ratio,
    random_seed=config.random_seed,
)
split_counts = pd.concat(
    [
        splits.train.assign(split="train"),
        splits.test.assign(split="test"),
    ]
).groupby(["split", "class_name", "label"]).size().rename("image_count")
display(split_counts.to_frame())

In [ ]:
sample_rows = splits.train.groupby("class_name", group_keys=False).head(2)
figure, axes = plt.subplots(1, len(sample_rows), figsize=(4 * len(sample_rows), 4))
for axis, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(f"Original: {row.class_name}")
    axis.axis("off")
figure.tight_layout()

In [ ]:
classifier_train = classifier_transform(training=True, image_size=config.high_resolution_size)
figure, axes = plt.subplots(1, len(sample_rows), figsize=(4 * len(sample_rows), 4))
for axis, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        transformed = classifier_train(image.convert("RGB"))
    axis.imshow(denormalize_imagenet(transformed).permute(1, 2, 0))
    axis.set_title(f"Classifier transform: {row.class_name}")
    axis.axis("off")
figure.tight_layout()

In [ ]:
pair_transform = SRGANPairTransform(
    training=True,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
figure, axes = plt.subplots(len(sample_rows), 2, figsize=(7, 3 * len(sample_rows)))
for row_axes, row in zip(axes, sample_rows.itertuples(), strict=True):
    with Image.open(row.filepath) as image:
        low_resolution, high_resolution = pair_transform(image.convert("RGB"))
    row_axes[0].imshow(denormalize_srgan(low_resolution).permute(1, 2, 0))
    row_axes[0].set_title(f"32×32 low resolution: {row.class_name}")
    row_axes[1].imshow(denormalize_srgan(high_resolution).permute(1, 2, 0))
    row_axes[1].set_title(f"128×128 target: {row.class_name}")
    for axis in row_axes:
        axis.axis("off")
figure.tight_layout()

---

# Phase 2 — Model A transfer learning


# Model A: MobileNetV2 transfer learning
Train a one-logit MobileNetV2 classifier on original 128×128 images. Validation rows are stratified from the persisted training CSV only; the reserved test CSV is used only after loading the best-validation checkpoint.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, load_splits
from applied_ai_midterm.evaluation import evaluate_classifier
from applied_ai_midterm.models import build_mobilenet_v2_classifier
from applied_ai_midterm.training import (
    create_train_validation_frames,
    fit_classifier,
    load_best_classifier,
    seed_everything,
    select_device,
)
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
train_frame, validation_frame = create_train_validation_frames(
    splits.train, validation_ratio=0.20, random_seed=config.random_seed
)
assert set(train_frame.filepath).isdisjoint(validation_frame.filepath)
assert set(splits.test.filepath).isdisjoint(train_frame.filepath)
assert set(splits.test.filepath).isdisjoint(validation_frame.filepath)
display(pd.DataFrame({"subset": ["train", "validation", "test"], "count": [len(train_frame), len(validation_frame), len(splits.test)]}))
print(f"Device: {device}")

In [ ]:
train_dataset = ImagePathDataset(
    train_frame, classifier_transform(training=True, image_size=config.high_resolution_size)
)
validation_dataset = ImagePathDataset(
    validation_frame, classifier_transform(training=False, image_size=config.high_resolution_size)
)
test_dataset = ImagePathDataset(
    splits.test, classifier_transform(training=False, image_size=config.high_resolution_size)
)
loader_kwargs = {"batch_size": config.classifier_batch_size, "num_workers": config.num_workers}
generator = torch.Generator().manual_seed(config.random_seed)
train_loader = DataLoader(train_dataset, shuffle=True, generator=generator, **loader_kwargs)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_kwargs)

In [ ]:
model = build_mobilenet_v2_classifier(freeze_features=True).to(device)
optimizer = torch.optim.Adam(
    (parameter for parameter in model.parameters() if parameter.requires_grad), lr=1e-3
)
checkpoint_path = Path("checkpoints/model_a_best.pt")
history = fit_classifier(
    model,
    train_loader,
    validation_loader,
    optimizer,
    checkpoint_path,
    epochs=config.classifier_epochs,
    device=device,
    config=asdict(config),
)

In [ ]:
history_frame = pd.DataFrame(
    {
        "epoch": [row["epoch"] for row in history],
        "training_loss": [row["training_loss"] for row in history],
        "validation_loss": [row["validation_loss"] for row in history],
    }
)
history_frame.plot(x="epoch", y=["training_loss", "validation_loss"], figsize=(8, 5))
plt.title("Model A loss")
plt.ylabel("BCEWithLogitsLoss")
plt.show()

In [ ]:
best_checkpoint = load_best_classifier(model, checkpoint_path, device)
criterion = nn.BCEWithLogitsLoss()
test_loss, test_metrics, test_labels, test_probabilities = evaluate_classifier(
    model, test_loader, criterion, device
)
display(pd.Series(test_metrics.as_dict(), name="Model A test metrics").to_frame())
print(f"Best validation epoch: {best_checkpoint['epoch']}")
print(f"Test BCE loss: {test_loss:.4f}")

---

# Phases 3–5 — SRGAN pretraining and adversarial training


# SRGAN training: reconstruction pretraining then adversarial training
Use only persisted training examples. First pretrain the generator with L1 reconstruction loss. Then begin adversarial generator/discriminator training. Both stages resume automatically and save checkpoints plus bicubic/generated/real sample grids every five epochs.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import SRGANPathDataset, load_splits
from applied_ai_midterm.models import Discriminator, Generator, GeneratorLoss, VGGPerceptualFeatures
from applied_ai_midterm.training import (
    pretrain_generator,
    resume_generator_pretraining,
    resume_latest_srgan,
    seed_everything,
    select_device,
    train_srgan,
)
from applied_ai_midterm.transforms import SRGANPairTransform, denormalize_srgan

In [ ]:
config = load_config(Path("configs/config.yaml"))
assert config.srgan_epochs >= 150
assert config.checkpoint_interval == 5
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
srgan_dataset = SRGANPathDataset(
    splits.train,  # Reserved test examples never enter either SRGAN stage.
    SRGANPairTransform(
        training=True,
        low_resolution_size=config.low_resolution_size,
        high_resolution_size=config.high_resolution_size,
    ),
)
loader_generator = torch.Generator().manual_seed(config.random_seed)
srgan_loader = DataLoader(
    srgan_dataset,
    batch_size=config.srgan_batch_size,
    shuffle=True,
    num_workers=config.num_workers,
    generator=loader_generator,
)
fixed_low_resolution, fixed_high_resolution = next(iter(srgan_loader))
sample_tensors = (fixed_low_resolution[:4], fixed_high_resolution[:4])
sample_dir = Path("artifacts/srgan_samples")
print(f"Training examples: {len(srgan_dataset)}; device: {device}")

## Stage 1 — generator reconstruction pretraining
This stage uses only L1 pixel/content reconstruction loss. The default below is 20 epochs and can resume from the latest `generator_pretrain_epoch_*.pt` file.

In [ ]:
generator = Generator().to(device)
pretrain_epochs = 20
pretrain_checkpoint_dir = Path("checkpoints/srgan_pretrain")
pretrain_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
pretrain_scheduler = torch.optim.lr_scheduler.StepLR(pretrain_optimizer, step_size=10, gamma=0.5)
pretrain_start_epoch, reconstruction_history, pretrain_resume_path = resume_generator_pretraining(
    pretrain_checkpoint_dir,
    generator=generator,
    optimizer=pretrain_optimizer,
    scheduler=pretrain_scheduler,
    device=device,
)
print(f"Resuming pretraining from {pretrain_resume_path}" if pretrain_resume_path else "Starting generator pretraining")
reconstruction_history = pretrain_generator(
    generator,
    srgan_loader,
    pretrain_optimizer,
    pretrain_checkpoint_dir,
    total_epochs=pretrain_epochs,
    start_epoch=pretrain_start_epoch,
    device=device,
    config=asdict(config),
    history=reconstruction_history,
    checkpoint_interval=config.checkpoint_interval,
    scheduler=pretrain_scheduler,
    sample_tensors=sample_tensors,
    sample_dir=sample_dir,
    sample_interval=5,
)

## Stage 2 — adversarial generator/discriminator training
The pretrained generator now uses pixel/content, adversarial, and optional VGG perceptual losses. Complete adversarial checkpoints resume both networks, both optimizers, both schedulers, and history.

In [ ]:
discriminator = Discriminator().to(device)
generator_optimizer = torch.optim.Adam(generator.parameters(), lr=1e-4, betas=(0.9, 0.999))
discriminator_optimizer = torch.optim.Adam(discriminator.parameters(), lr=1e-4, betas=(0.9, 0.999))
generator_scheduler = torch.optim.lr_scheduler.StepLR(generator_optimizer, step_size=50, gamma=0.5)
discriminator_scheduler = torch.optim.lr_scheduler.StepLR(discriminator_optimizer, step_size=50, gamma=0.5)
use_vgg_perceptual_loss = True
perceptual_features = VGGPerceptualFeatures().to(device) if use_vgg_perceptual_loss else None
generator_loss = GeneratorLoss(perceptual_features=perceptual_features).to(device)
adversarial_checkpoint_dir = Path("checkpoints/srgan")
start_epoch, history, adversarial_resume_path = resume_latest_srgan(
    adversarial_checkpoint_dir,
    generator=generator,
    discriminator=discriminator,
    generator_optimizer=generator_optimizer,
    discriminator_optimizer=discriminator_optimizer,
    generator_scheduler=generator_scheduler,
    discriminator_scheduler=discriminator_scheduler,
    device=device,
)
print(f"Resuming adversarial training from {adversarial_resume_path}" if adversarial_resume_path else "Starting adversarial training from pretrained generator")
history = train_srgan(
    generator,
    discriminator,
    srgan_loader,
    generator_optimizer,
    discriminator_optimizer,
    generator_loss,
    adversarial_checkpoint_dir,
    total_epochs=config.srgan_epochs,
    start_epoch=start_epoch,
    device=device,
    config=asdict(config),
    history=history,
    checkpoint_interval=config.checkpoint_interval,
    generator_scheduler=generator_scheduler,
    discriminator_scheduler=discriminator_scheduler,
    sample_tensors=sample_tensors,
    sample_dir=sample_dir,
    sample_interval=5,
)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(reconstruction_history)
axes[0].set(title="Generator pretraining", xlabel="Epoch", ylabel="L1 reconstruction loss")
pd.DataFrame(history).plot(ax=axes[1])
axes[1].set(title="Adversarial SRGAN training", xlabel="Epoch", ylabel="Loss")
figure.tight_layout()

In [ ]:
generator.eval()
with torch.inference_mode():
    generated = generator(fixed_low_resolution[:1].to(device)).cpu()[0]
figure, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(denormalize_srgan(fixed_low_resolution[0]).permute(1, 2, 0))
axes[0].set_title("32×32 input")
axes[1].imshow(denormalize_srgan(generated).permute(1, 2, 0))
axes[1].set_title("Generated 128×128")
axes[2].imshow(denormalize_srgan(fixed_high_resolution[0]).permute(1, 2, 0))
axes[2].set_title("Real 128×128 target")
for axis in axes:
    axis.axis("off")
figure.tight_layout()

---

# Phase 5 — Generate SRGAN training images


# Generate Model B training images
Select the adversarial checkpoint with the lowest recorded total generator loss, process only persisted training images, preserve class folders, and save a complete source manifest. The reserved test split is not generated or accessed here.

In [ ]:
from pathlib import Path

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import load_splits
from applied_ai_midterm.generation import generate_training_images
from applied_ai_midterm.models import Generator
from applied_ai_midterm.training import load_best_generator, select_device
from applied_ai_midterm.visualization import plot_srgan_comparisons

In [ ]:
config = load_config(Path("configs/config.yaml"))
device = select_device()
splits = load_splits(Path("data/splits"))
generator = Generator()
checkpoint_path, checkpoint = load_best_generator(
    generator, Path("checkpoints/srgan"), device
)
print(f"Selected epoch {checkpoint['epoch']} from {checkpoint_path}")
print(f"Generator selection loss: {checkpoint.get('generator_selection_loss')}")

In [ ]:
# Only splits.train is passed. Generated files are grouped by detected class_name.
manifest = generate_training_images(
    generator,
    splits.train,
    Path("data/generated"),
    Path("data/generated/manifest.csv"),
    checkpoint_path,
    device,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
display(manifest.head())
display(manifest.groupby(["class_name", "label"]).size().rename("generated_count").to_frame())
assert len(manifest) == len(splits.train)
assert set(manifest.source_filepath).isdisjoint(set(splits.test.filepath))

In [ ]:
comparison_figure = plot_srgan_comparisons(
    generator,
    splits.train,
    device,
    output_path=Path("artifacts/srgan_generation_comparison.png"),
    max_samples=4,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
comparison_figure

---

# Phase 6 — Model B transfer learning


# Model B: MobileNetV2 trained on SRGAN images
Train a fresh classifier using the same architecture and training settings as Model A. The only change is that Model B's training and validation inputs come from the validated SRGAN-generated training manifest.

In [ ]:
from dataclasses import asdict
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, load_splits
from applied_ai_midterm.generation import load_generated_training_manifest
from applied_ai_midterm.models import build_mobilenet_v2_classifier
from applied_ai_midterm.training import (
    create_train_validation_frames,
    fit_classifier,
    seed_everything,
    select_device,
)
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
generated_frame = load_generated_training_manifest(
    Path("data/generated/manifest.csv"), splits.train
)
train_frame, validation_frame = create_train_validation_frames(
    generated_frame, validation_ratio=0.20, random_seed=config.random_seed
)
assert len(train_frame) + len(validation_frame) == len(splits.train)
display(pd.DataFrame({"subset": ["generated train", "generated validation"], "count": [len(train_frame), len(validation_frame)]}))

In [ ]:
# These transformations, loader settings, and seed match Model A exactly.
train_dataset = ImagePathDataset(
    train_frame, classifier_transform(training=True, image_size=config.high_resolution_size)
)
validation_dataset = ImagePathDataset(
    validation_frame, classifier_transform(training=False, image_size=config.high_resolution_size)
)
loader_kwargs = {"batch_size": config.classifier_batch_size, "num_workers": config.num_workers}
loader_generator = torch.Generator().manual_seed(config.random_seed)
train_loader = DataLoader(train_dataset, shuffle=True, generator=loader_generator, **loader_kwargs)
validation_loader = DataLoader(validation_dataset, shuffle=False, **loader_kwargs)

In [ ]:
# Fresh MobileNetV2, same frozen features, Adam learning rate, epochs, and BCEWithLogitsLoss workflow as Model A.
model_b = build_mobilenet_v2_classifier(freeze_features=True).to(device)
optimizer = torch.optim.Adam(
    (parameter for parameter in model_b.parameters() if parameter.requires_grad), lr=1e-3
)
history = fit_classifier(
    model_b,
    train_loader,
    validation_loader,
    optimizer,
    Path("checkpoints/model_b_best.pt"),
    epochs=config.classifier_epochs,
    device=device,
    config={**asdict(config), "model": "Model B", "training_source": "SRGAN manifest"},
)

In [ ]:
history_frame = pd.DataFrame(
    {
        "epoch": [row["epoch"] for row in history],
        "training_loss": [row["training_loss"] for row in history],
        "validation_loss": [row["validation_loss"] for row in history],
    }
)
history_frame.plot(x="epoch", y=["training_loss", "validation_loss"], figsize=(8, 5))
plt.title("Model B loss")
plt.ylabel("BCEWithLogitsLoss")
plt.show()

---

# Phase 7 — Final measured comparison


# Phase 7 — Final comparison
Load both best-validation classifiers and evaluate them on the same ordered reserved test examples. Model A receives original images resized to 128×128. Model B receives on-the-fly SRGAN versions of those same sources; test images are never saved or used for training. The resulting table contains measured values only after this notebook runs.

In [ ]:
from pathlib import Path

import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision.models import MobileNet_V2_Weights

from applied_ai_midterm.config import load_config
from applied_ai_midterm.data import ImagePathDataset, SRGANClassifierDataset, load_splits
from applied_ai_midterm.evaluation import build_model_comparison_table, evaluate_classifier
from applied_ai_midterm.models import Generator, build_mobilenet_v2_classifier
from applied_ai_midterm.training import load_best_classifier, load_best_generator, seed_everything, select_device
from applied_ai_midterm.transforms import classifier_transform

In [ ]:
config = load_config(Path("configs/config.yaml"))
seed_everything(config.random_seed)
device = select_device()
splits = load_splits(Path("data/splits"))
generator = Generator()
generator_checkpoint_path, _ = load_best_generator(
    generator, Path("checkpoints/srgan"), device
)
print(f"Using generator: {generator_checkpoint_path}")

In [ ]:
# Both loaders retain identical persisted test-row order and labels.
model_a_test_dataset = ImagePathDataset(
    splits.test, classifier_transform(training=False, image_size=config.high_resolution_size)
)
model_b_test_dataset = SRGANClassifierDataset(
    splits.test,
    generator,
    device,
    low_resolution_size=config.low_resolution_size,
    high_resolution_size=config.high_resolution_size,
)
model_a_test_loader = DataLoader(
    model_a_test_dataset, batch_size=config.classifier_batch_size, shuffle=False, num_workers=config.num_workers
)
# Generator inference occurs inside this dataset, so workers remain zero for device safety.
model_b_test_loader = DataLoader(
    model_b_test_dataset, batch_size=config.classifier_batch_size, shuffle=False, num_workers=0
)

In [ ]:
# weights=None avoids unnecessary downloads because complete trained states are loaded next.
model_a = build_mobilenet_v2_classifier(weights=None, freeze_features=True)
model_b = build_mobilenet_v2_classifier(weights=None, freeze_features=True)
load_best_classifier(model_a, Path("checkpoints/model_a_best.pt"), device)
load_best_classifier(model_b, Path("checkpoints/model_b_best.pt"), device)
criterion = nn.BCEWithLogitsLoss()
_, model_a_metrics, model_a_labels, model_a_probabilities = evaluate_classifier(
    model_a, model_a_test_loader, criterion, device
)
_, model_b_metrics, model_b_labels, model_b_probabilities = evaluate_classifier(
    model_b, model_b_test_loader, criterion, device
)
assert (model_a_labels == model_b_labels).all()
assert len(model_a_probabilities) == len(model_b_probabilities) == len(splits.test)

In [ ]:
comparison_table = build_model_comparison_table(
    model_a_metrics,
    model_b_metrics,
    output_path=Path("artifacts/final_model_comparison.csv"),
)
display(comparison_table.style.format({column: "{:.4f}" for column in comparison_table.columns if column != "Model"}))